# IWTC Tools: Narrative Indexing

# Pre-Build

## Phase P0: Parameters

Select the world repository descriptor and the index version you want to explore.

This notebook does not regenerate artifacts.  
It simply loads the existing graph artifacts for the chosen version.

In [1]:
# -------------------------------------------------------------------
# Phase P0: Parameters
# -------------------------------------------------------------------
LAST_PHASE_RUN = "P0"

# Absolute path to the world_repository.yml descriptor.
WORLD_REPOSITORY_DESCRIPTOR = (
    "/Users/charissophia/obsidian/Iron Wolf Trading Company/_meta/descriptors/world_repository.yml"
)

# Artifact version to load (must match previously generated artifacts).
# In v0, graph artifacts are versioned using the same INDEX_VERSION as the index artifacts they were built from.
INDEX_VERSION = "V0"

# Internal run metadata (do not edit)
from datetime import datetime
print(f"Notebook run initialized at: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
del datetime

Notebook run initialized at: 2026-04-09 13:38


## Phase P1: Load and validate world descriptor

The world descriptor defines where the repository’s artifacts live.

This phase:

• loads the descriptor  
• validates that the expected folders exist  
• resolves canonical paths used by the rest of the notebook

If this phase fails, the repository layout or descriptor likely needs correction.

In [2]:
# Phase P1: Load and validate world repository descriptor
LAST_PHASE_RUN = "P1"

from pathlib import Path
import yaml

errors = []
warnings = []

# --- Load descriptor file ---
descriptor_path = Path(WORLD_REPOSITORY_DESCRIPTOR)

if not descriptor_path.exists():
    raise FileNotFoundError(
        "World repository descriptor file was not found.\n"
        f"Path provided:\n  {descriptor_path}\n\n"
        "What to do:\n"
        "- Confirm the file exists at this location or fix WORLD_REPOSITORY_DESCRIPTOR in Phase 0\n"
        "- If you just edited Phase 0, rerun Phase 0 and then rerun this cell\n"
    )

try:
    with descriptor_path.open("r", encoding="utf-8") as f:
        world_repo = yaml.safe_load(f)
except Exception:
    raise ValueError(
        "The world repository descriptor could not be read.\n"
        "This usually indicates a YAML formatting problem.\n\n"
        f"File:\n  {descriptor_path}\n\n"
        "What to do:\n"
        "- Compare the file against the example world_repository.yml\n"
        "- Paste the contents into https://www.yamllint.com/\n"
        "- Fix any reported issues, save the file, and rerun this cell"
    )

if not isinstance(world_repo, dict):
    raise ValueError(
        "World repository descriptor structure is not usable.\n"
        "The file must be a YAML mapping (top-level `name: value` entries).\n"
    )

print(f"World repository descriptor loaded successfully: {descriptor_path.name}")

# --- Extract required entries ---
WORLD_ROOT_RAW = world_repo.get("world_root")

drafts_block = world_repo.get("working_drafts")
DRAFTS_RAW = drafts_block.get("path") if isinstance(drafts_block, dict) else None

indexes_block = world_repo.get("indexes")
INDEXES_RAW = indexes_block.get("path") if isinstance(indexes_block, dict) else None

vocab = world_repo.get("vocabulary") or {}
ENTITIES_RAW = vocab.get("entities")
RELATIONSHIPS_RAW = vocab.get("relationships")  # optional for now

if not WORLD_ROOT_RAW:
    errors.append("Missing required entry: world_root")
if not DRAFTS_RAW:
    errors.append("Missing required entry: working_drafts.path")
if not INDEXES_RAW:
    errors.append("Missing required entry: indexes.path")
if not ENTITIES_RAW:
    errors.append("Missing required entry: vocabulary.entities")

if errors:
    raise ValueError(
        "World repository descriptor is missing required entries:\n- "
        + "\n- ".join(errors)
        + "\n\nWhat to do:\n"
          "- Edit your world_repository.yml and add/fix the missing entries\n"
          "- Save the file and rerun this cell"
    )

# ------------------------------------------------------------------
# Published outputs (initialize up front for later phases)
# ------------------------------------------------------------------
WORLD_ROOT = None

WORKING_DRAFTS_PATH = None
WORKING_DRAFTS_RELPATH = None

INDEXES_PATH = None
INDEXES_RELPATH = None

VOCAB_ENTITIES_PATH = None
VOCAB_ENTITIES_RELPATH = None

VOCAB_RELATIONSHIPS_PATH = None
VOCAB_RELATIONSHIPS_RELPATH = None

# --- Validate and resolve world_root ---
WORLD_ROOT = Path(WORLD_ROOT_RAW)

if str(WORLD_ROOT).startswith("~"):
    errors.append("world_root: '~' is not allowed. Use a full absolute path.")
elif not WORLD_ROOT.is_absolute():
    errors.append("world_root must be an absolute path (starts with / on macOS/Linux, or C:\\ on Windows).")
elif not WORLD_ROOT.is_dir():
    errors.append(f"world_root must be an existing directory: {WORLD_ROOT}")
else:
    WORLD_ROOT = WORLD_ROOT.resolve()

if errors:
    raise ValueError("Descriptor path validation failed:\n- " + "\n- ".join(errors))

def _resolve_under_world_root(raw_path: str, label: str):
    if raw_path is None or str(raw_path).strip() == "":
        return None, None

    p = Path(str(raw_path))

    if str(p).startswith("~"):
        errors.append(f"{label}: '~' is not allowed: {raw_path}")
        return None, None

    if not p.is_absolute():
        p = WORLD_ROOT / p
    p = p.resolve()

    try:
        rel = str(p.relative_to(WORLD_ROOT))
    except Exception:
        rel = str(p)

    return p, rel

# --- Resolve and validate working_drafts path (required, directory, writable) ---
WORKING_DRAFTS_PATH, WORKING_DRAFTS_RELPATH = _resolve_under_world_root(DRAFTS_RAW, "working_drafts.path")

if WORKING_DRAFTS_PATH is None:
    errors.append("working_drafts.path: missing or invalid.")
else:
    if not WORKING_DRAFTS_PATH.exists():
        errors.append(f"working_drafts.path: path does not exist: {WORKING_DRAFTS_PATH}")
    elif not WORKING_DRAFTS_PATH.is_dir():
        errors.append(f"working_drafts.path: must be a directory: {WORKING_DRAFTS_PATH}")
    else:
        probe = WORKING_DRAFTS_PATH / ".iwtc_tools_write_probe.tmp"
        try:
            probe.write_text("test", encoding="utf-8")
        except Exception as e:
            errors.append(f"working_drafts.path: not writable: {WORKING_DRAFTS_PATH} ({type(e).__name__})")
        finally:
            try:
                if probe.exists():
                    probe.unlink()
            except Exception:
                pass
        del probe

# --- Resolve and validate indexes path (required, directory) ---
INDEXES_PATH, INDEXES_RELPATH = _resolve_under_world_root(INDEXES_RAW, "indexes.path")

if INDEXES_PATH is None:
    errors.append("indexes.path: missing or invalid.")
else:
    if not INDEXES_PATH.exists():
        errors.append(f"indexes.path: path does not exist: {INDEXES_PATH}")
    elif not INDEXES_PATH.is_dir():
        errors.append(f"indexes.path: must be a directory: {INDEXES_PATH}")

# --- Resolve and validate vocabulary.entities (required, file) ---
VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH = _resolve_under_world_root(ENTITIES_RAW, "vocabulary.entities")

if VOCAB_ENTITIES_PATH is None:
    errors.append("vocabulary.entities: missing or invalid.")
else:
    if not VOCAB_ENTITIES_PATH.exists():
        errors.append(f"vocabulary.entities: file does not exist: {VOCAB_ENTITIES_PATH}")
    elif VOCAB_ENTITIES_PATH.is_dir():
        errors.append(f"vocabulary.entities: must be a file, got directory: {VOCAB_ENTITIES_PATH}")

# --- Resolve vocabulary.relationships (optional for now) ---
if RELATIONSHIPS_RAW:
    VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH = _resolve_under_world_root(
        RELATIONSHIPS_RAW, "vocabulary.relationships"
    )

    if VOCAB_RELATIONSHIPS_PATH is not None:
        if VOCAB_RELATIONSHIPS_PATH.exists() and VOCAB_RELATIONSHIPS_PATH.is_dir():
            warnings.append(
                f"vocabulary.relationships: must be a file, got directory: {VOCAB_RELATIONSHIPS_PATH}"
            )
            VOCAB_RELATIONSHIPS_PATH = None
            VOCAB_RELATIONSHIPS_RELPATH = None
        elif not VOCAB_RELATIONSHIPS_PATH.exists():
            warnings.append(
                f"vocabulary.relationships: file does not yet exist: {VOCAB_RELATIONSHIPS_PATH}"
            )

if errors:
    raise ValueError("Descriptor path validation failed:\n- " + "\n- ".join(errors))

print("Descriptor paths are usable for this notebook.")
print(f"world_root: {WORLD_ROOT}")
print(f"working_drafts: {WORKING_DRAFTS_RELPATH}")
print(f"indexes: {INDEXES_RELPATH}")
print(
    f"vocabulary.entities: {VOCAB_ENTITIES_RELPATH} "
    f"(exists={VOCAB_ENTITIES_PATH.exists() if VOCAB_ENTITIES_PATH else False})"
)
print(
    f"vocabulary.relationships: {VOCAB_RELATIONSHIPS_RELPATH} "
    f"(exists={VOCAB_RELATIONSHIPS_PATH.exists() if VOCAB_RELATIONSHIPS_PATH else False})"
)

if warnings:
    print("\nWarnings:")
    for w in warnings:
        print(f"- {w}")

# cleanup
del yaml, Path
del descriptor_path, world_repo, drafts_block, indexes_block, vocab
del WORLD_ROOT_RAW, DRAFTS_RAW, INDEXES_RAW, ENTITIES_RAW, RELATIONSHIPS_RAW
del warnings, errors, f
del _resolve_under_world_root

World repository descriptor loaded successfully: world_repository.yml
Descriptor paths are usable for this notebook.
world_root: /Users/charissophia/obsidian/Iron Wolf Trading Company
working_drafts: _local/machine_wip
indexes: _meta/indexes
vocabulary.entities: _meta/indexes/vocab_entities.csv (exists=True)
vocabulary.relationships: _meta/indexes/world_relationships.csv (exists=True)


## Phase P2: Load required index and vocabulary artifacts

This phase verifies that the required generated index artifacts and required vocabulary files exist and can be loaded.

- Resolve expected versioned index filenames from `INDEX_VERSION`
- Load required generated index CSVs from `indexes.path`
- Load required vocabulary CSVs from descriptor-defined paths
- Verify required columns are present

If any artifact is missing or malformed, the notebook stops with instructions to correct the inputs.

No files are modified in this phase.

In [3]:
# Phase P2: Load required index and vocabulary artifacts
LAST_PHASE_RUN = "P2"

import pandas as pd
import itertools  # not used here, but needed during bootstrapping

errors = []

# Normalize INDEX_VERSION into the on-disk suffix
INDEX_VERSION_SUFFIX = f"v{str(INDEX_VERSION).lower().lstrip('v')}"

# Required generated index artifacts (versioned, loaded from indexes.path)
required_indexes = {
    "chunk_to_entities": f"index_chunk_to_entities_{INDEX_VERSION_SUFFIX}.csv",
    "source_files": f"index_source_files_{INDEX_VERSION_SUFFIX}.csv",
}

INDEX_FILES = {}

for key, fname in required_indexes.items():
    p = (INDEXES_PATH / fname).resolve()
    INDEX_FILES[key] = p
    if not p.exists():
        errors.append(f"Missing required artifact: {fname}\n  Expected at: {p}")

# Required vocabulary artifacts (descriptor-defined, not versioned)
if VOCAB_ENTITIES_PATH is None:
    errors.append("Missing required descriptor-resolved path: vocabulary.entities")
elif not VOCAB_ENTITIES_PATH.exists():
    errors.append(f"Missing required vocabulary file: {VOCAB_ENTITIES_PATH}")

if errors:
    raise FileNotFoundError(
        "Phase P2 cannot proceed because required artifacts are missing.\n\n"
        + "\n\n".join(errors)
        + "\n\nWhat to do:\n"
          "- Ensure raw source indexing has been executed\n"
          "- Confirm the required index CSV files exist under indexes.path\n"
          "- Confirm vocabulary.entities is correctly defined in world_repository.yml\n"
          "- Then rerun Phase P2"
    )

# Load CSVs
DF_CHUNK_TO_ENTITIES = pd.read_csv(INDEX_FILES["chunk_to_entities"])
DF_SOURCE_FILES = pd.read_csv(INDEX_FILES["source_files"])
DF_VOCAB_ENTITIES = pd.read_csv(VOCAB_ENTITIES_PATH)

# Validate minimal required columns
expected_cols = {
    "DF_CHUNK_TO_ENTITIES": {
        "chunk_id", "source_id", "source_type", "relpath",
        "chunk_start_line", "chunk_end_line",
        "entity_ids", "entity_count"
    },
    "DF_SOURCE_FILES": {"source_id", "relpath", "source_type"},
    "DF_VOCAB_ENTITIES": {"entity_id", "canonical_name", "entity_type", "layer"},
}

for df_name, cols in expected_cols.items():
    df = globals()[df_name]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        errors.append(f"{df_name}: missing expected columns: {missing}")

if errors:
    raise ValueError(
        "One or more artifacts were loaded but do not match expected structure.\n- "
        + "\n- ".join(errors)
        + "\n\nWhat to do:\n"
          "- Confirm you are using the expected canonical CSV artifacts\n"
          "- If necessary regenerate the index CSVs via IWTC_Raw_Source_Indexing.ipynb\n"
          "- Confirm vocabulary.entities points to the correct file in world_repository.yml"
    )

print("Phase P2 OK: required artifacts loaded.")
print(f"indexes.path: {INDEXES_PATH}")
print(f"index version: {INDEX_VERSION_SUFFIX}")

print("\nLoaded tables:")
print(f"- DF_CHUNK_TO_ENTITIES: {len(DF_CHUNK_TO_ENTITIES):>8} rows")
print(f"- DF_SOURCE_FILES:      {len(DF_SOURCE_FILES):>8} rows")
print(f"- DF_VOCAB_ENTITIES:    {len(DF_VOCAB_ENTITIES):>8} rows")

# cleanup
del errors, required_indexes, key, fname, p, df_name, df, cols, missing
del expected_cols, INDEX_FILES, INDEX_VERSION_SUFFIX

Phase P2 OK: required artifacts loaded.
indexes.path: /Users/charissophia/obsidian/Iron Wolf Trading Company/_meta/indexes
index version: v0

Loaded tables:
- DF_CHUNK_TO_ENTITIES:     1144 rows
- DF_SOURCE_FILES:           130 rows
- DF_VOCAB_ENTITIES:         402 rows


# Narrative Bootstrapping

## B1: Getting oriented in the data

1. look at `IWTC session notes 51-100.md`
2. find the first few chunks for that file and compare what I see

In [18]:
# Phase B1: inspect header data
LAST_PHASE_RUN = "B1"

# Find first few chunks for this source file
sample_source = "_local/session_notes/IWTC session notes 51-100.md"


# ------------------------
from pathlib import Path
from itertools import islice

# get the first few chunks for this file
df_sample = (
    DF_CHUNK_TO_ENTITIES
    .loc[DF_CHUNK_TO_ENTITIES["source_id"] == sample_source]
    .sort_values("chunk_start_line")
    .head(2)
)

print(f"First few chunks of {sample_source}\n") 
display(df_sample[[
    "chunk_id", "source_id",
    "chunk_start_line","chunk_end_line",
    "entity_count"
]])

# load source text
print("\nSource text for those lines:\n") 

sample_path = WORLD_ROOT / sample_source
for _, row in df_sample.iterrows():
    start_line = int(row["chunk_start_line"])
    end_line = int(row["chunk_end_line"])

    with sample_path.open("r", encoding="utf-8") as f:
        lines = list(islice(f, start_line - 1, end_line))

    print("=" * 80)
    print(f"{sample_source} | chunk {row['chunk_id']} | lines {start_line}-{end_line}")
    print("-" * 80)
    print("".join(lines))
    print()

First few chunks of _local/session_notes/IWTC session notes 51-100.md



,chunk_id,source_id,chunk_start_line,chunk_end_line,entity_count
917,169840,_local/session_notes/IWTC session notes 51-100.md,3,24,11
918,169841,_local/session_notes/IWTC session notes 51-100.md,25,53,12



Source text for those lines:

_local/session_notes/IWTC session notes 51-100.md | chunk 169840 | lines 3-24
--------------------------------------------------------------------------------
## Session 51: Sat 2021-07-03

Alivyre returns to Crafthold with the 3 horses abandoned by the Spire folk when they banished home and with the 225gp from Soren who she meets on the road between Brightfield and Crafthold. She’s surprised they haven’t rescued the missing orcs. They weren’t aware. We plan how to deal with spiders and fire seems the best approach. Extreme cold might work. 

Alivyre, Unala, Jane, Liacyne, Bre, Henry, Mellow  
![][image1]Day and a half later, arrive at the spiders. Henry uses the sonic screwdriver to agro the spiders and Jane casts a wall of fire. Pet and the winter wolves plow with their breath. AOEs clear through the spiders with almost no damage to the party.

Recover 32 orc bodies, 12 wolves, 3 bats. Jane finds a cave with 3 orcs still alive, but unconscious, one of t

That tells us a chunk = a session in session_notes. The first line is what was identified as a header. Let's look at just header lines, using the same sample as before.

In [23]:
print("First line of first 15 chunks:\n")

# get the first few chunks for this file
df_sample = (
    DF_CHUNK_TO_ENTITIES
    .loc[DF_CHUNK_TO_ENTITIES["source_id"] == sample_source]
    .sort_values("chunk_start_line")
#    .head(15)
)

for _, row in df_sample.iterrows():
    start_line = int(row["chunk_start_line"])

    with sample_path.open("r", encoding="utf-8") as f:
        line = next(islice(f, start_line - 1, start_line))

    print(f"{row['chunk_id']} (line {start_line}): {line.strip()}")

First line of first 15 chunks:

169840 (line 3): ## Session 51: Sat 2021-07-03
169841 (line 25): ## Session 52: Sat 2021-07-17
169842 (line 54): ### Notes for Spartan’s caster character
169843 (line 60): ## Session 53: Sat 2021-07-24
169844 (line 90): ## Session 54: Sat 2021-08-07
169845 (line 114): ## Session 55: Sat 2021-08-21
169846 (line 127): ### Adventure to get Poppy:
169847 (line 141): ## Session 56: Sat 2021-08-28
169848 (line 156): ## Session 57: Sat 2021-09-11
169850 (line 230): ### Spire
169851 (line 244): ### Plane of Elemental Spirits
169852 (line 270): ### Credus
169854 (line 284): ### Crafthold \- Enforcers looking for Credus \- again
169855 (line 298): #### Repeat of session 38:
169856 (line 320): #### This session notes:
169859 (line 338): ### Notes for Bysickle for running the drider party:
169860 (line 364): ### East Road Party \- Drider encounter
169862 (line 405): ## Session 61: Sat 2021-10-23
169863 (line 452): ## Session 62: Sat 2021-10-30
169864 (line 474): ## 

## Phase B2: When did we rescue Blackburn?

### Find chunks for an entity

In [19]:
# -------------------------------------------------------------------
# Phase B2.1: Find chunks for an entity (start)
# -------------------------------------------------------------------
LAST_PHASE_RUN = "B2"

# Example query
search_term = "Blackburn"

# find matching entity_ids from vocab
df_entity_match = (
    DF_VOCAB_ENTITIES
    .loc[DF_VOCAB_ENTITIES["canonical_name"].str.contains(search_term, case=False, na=False)]
)

display(df_entity_match[["entity_id", "canonical_name"]])

# get the entity_id
entity_id = df_entity_match.iloc[0]["entity_id"]

# find chunks where it appears
df_chunks = (
    DF_CHUNK_TO_ENTITIES
    .loc[
        (DF_CHUNK_TO_ENTITIES["entity_ids"].str.contains(entity_id, na=False)) &
        (DF_CHUNK_TO_ENTITIES["source_type"] == "session_notes")
    ]
    .sort_values(["source_id", "chunk_start_line"])
)

display(df_chunks[[
    "chunk_id",
    "source_id",
    "chunk_start_line",
    "chunk_end_line",
    "entity_count"
]].head(10))

print("\nRelevant chunks:\n")

for _, row in df_chunks.iterrows():
    source = row["source_id"]
    start_line = int(row["chunk_start_line"])
    end_line = int(row["chunk_end_line"])

    path = WORLD_ROOT / source

    with path.open("r", encoding="utf-8") as f:
        lines = list(islice(f, start_line - 1, end_line))

    print("=" * 80)
    print(f"{source} | chunk {row['chunk_id']} | lines {start_line}-{end_line}")
    print("-" * 80)
    print("".join(lines))
    print()

,entity_id,canonical_name


IndexError: single positional indexer is out-of-bounds

### Split chunk into ordered text blocks

In [9]:
# -------------------------------------------------------------------
# split one chunk into ordered text blocks
# -------------------------------------------------------------------
LAST_PHASE_RUN = "B3"

from itertools import islice

sample_source = "_local/session_notes/IWTC session notes 1-50.md"
sample_chunk_id = 169539

row = (
    DF_CHUNK_TO_ENTITIES
    .loc[DF_CHUNK_TO_ENTITIES["chunk_id"] == sample_chunk_id]
    .iloc[0]
)

start_line = int(row["chunk_start_line"])
end_line = int(row["chunk_end_line"])
sample_path = WORLD_ROOT / sample_source

with sample_path.open("r", encoding="utf-8") as f:
    chunk_lines = list(islice(f, start_line - 1, end_line))

chunk_text = "".join(chunk_lines)

blocks = []
current_block = []

for line in chunk_text.splitlines():
    if line.strip():
        current_block.append(line)
    else:
        if current_block:
            blocks.append("\n".join(current_block))
            current_block = []

if current_block:
    blocks.append("\n".join(current_block))

print(f"Chunk {sample_chunk_id} split into {len(blocks)} blocks.\n")

for i, block in enumerate(blocks, start=1):
    print("=" * 80)
    print(f"Block {i}")
    print("-" * 80)
    print(block)
    print()

Chunk 169539 split into 9 blocks.

Block 1
--------------------------------------------------------------------------------
## Session 21: Sat 2020-08-22 \- Travel Tribulations

Block 2
--------------------------------------------------------------------------------
Faeryne feels a compulsion to find the Foreclaimer Site. She reaches out to Alivyre one evening who goes to the army captain to get some help. Bre and Zevren assigned. Estimate 4 days travel.

Block 3
--------------------------------------------------------------------------------
Evening watch at Brightfields: (Glimmer)

Block 4
--------------------------------------------------------------------------------
- Bre/Zevren  
- Faeryne/Luminia  
- Alivyre/Jane \<- pre-dawn notice the mist and get the party out of Brightfields

Block 5
--------------------------------------------------------------------------------
They get moving and out of Brightfields in the pre-dawn. Luminia can phase in/out of the glimmer. Attacked by ban

### Classify text blocks by surface form

In [5]:
# -------------------------------------------------------------------
# Phase B4: classify text blocks by surface form
# -------------------------------------------------------------------
LAST_PHASE_RUN = "B4"

import re

classified_blocks = []

for i, block in enumerate(blocks, start=1):
    lines = block.splitlines()
    first_line = lines[0].strip()
    nonempty_lines = [ln.strip() for ln in lines if ln.strip()]

    if re.match(r"^\s*#{1,6}\s+", first_line):
        block_type = "header"

    elif all(ln.startswith("- ") for ln in nonempty_lines):
        joined = " ".join(nonempty_lines).lower()

        if re.search(r"\bxp\b|\bcr\d*\b|\(\d+\s*xp", joined):
            block_type = "combat_stats"
        elif re.search(r"\bgp\b|\bdagger\b|\bscimitar\b|\bhorses?\b|\bfood\b|\bdrink\b|\bhousehold\b", joined):
            block_type = "loot"
        else:
            block_type = "list"

    elif re.search(r"/r\s*\(", block):
        block_type = "xp_accounting"

    else:
        block_type = "narrative"

    classified_blocks.append(
        {
            "block_num": i,
            "block_type": block_type,
            "text": block,
        }
    )

for item in classified_blocks:
    print("=" * 80)
    print(f"Block {item['block_num']} | {item['block_type']}")
    print("-" * 80)
    print(item["text"])
    print()

Block 1 | header
--------------------------------------------------------------------------------
## Session 21: Sat 2020-08-22 \- Travel Tribulations

Block 2 | narrative
--------------------------------------------------------------------------------
Faeryne feels a compulsion to find the Foreclaimer Site. She reaches out to Alivyre one evening who goes to the army captain to get some help. Bre and Zevren assigned. Estimate 4 days travel.

Block 3 | narrative
--------------------------------------------------------------------------------
Evening watch at Brightfields: (Glimmer)

Block 4 | list
--------------------------------------------------------------------------------
- Bre/Zevren  
- Faeryne/Luminia  
- Alivyre/Jane \<- pre-dawn notice the mist and get the party out of Brightfields

Block 5 | narrative
--------------------------------------------------------------------------------
They get moving and out of Brightfields in the pre-dawn. Luminia can phase in/out of the glimmer

### Filter to likely event candidates

In [7]:
event_blocks = [
    {
        "source_id": sample_source,
        "chunk_id": sample_chunk_id,
        "block_num": b["block_num"],
        "block_type": b["block_type"],
        "text": b["text"],
    }
    for b in classified_blocks
    if b["block_type"] in {"narrative", "header"}
]

for b in event_blocks:
    print("=" * 80)
    print(
        f"{b['source_id']} | chunk {b['chunk_id']} | "
        f"block {b['block_num']} | {b['block_type']}"
    )
    print("-" * 80)
    print(b["text"])
    print()

_local/session_notes/IWTC session notes 1-50.md | chunk 169539 | block 1 | header
--------------------------------------------------------------------------------
## Session 21: Sat 2020-08-22 \- Travel Tribulations

_local/session_notes/IWTC session notes 1-50.md | chunk 169539 | block 2 | narrative
--------------------------------------------------------------------------------
Faeryne feels a compulsion to find the Foreclaimer Site. She reaches out to Alivyre one evening who goes to the army captain to get some help. Bre and Zevren assigned. Estimate 4 days travel.

_local/session_notes/IWTC session notes 1-50.md | chunk 169539 | block 3 | narrative
--------------------------------------------------------------------------------
Evening watch at Brightfields: (Glimmer)

_local/session_notes/IWTC session notes 1-50.md | chunk 169539 | block 5 | narrative
--------------------------------------------------------------------------------
They get moving and out of Brightfields in the pre

### Inspect event-candidate blocks across Blackburn chunks

In [10]:
import re
from itertools import islice

print(f"Blackburn-related session_notes chunks: {len(df_chunks)}\n")

for _, row in df_chunks.iterrows():
    source_id = row["source_id"]
    chunk_id = int(row["chunk_id"])
    start_line = int(row["chunk_start_line"])
    end_line = int(row["chunk_end_line"])

    source_path = WORLD_ROOT / source_id

    with source_path.open("r", encoding="utf-8") as f:
        chunk_lines = list(islice(f, start_line - 1, end_line))

    chunk_text = "".join(chunk_lines)

    blocks = []
    current_block = []

    for line in chunk_text.splitlines():
        if line.strip():
            current_block.append(line)
        else:
            if current_block:
                blocks.append("\n".join(current_block))
                current_block = []

    if current_block:
        blocks.append("\n".join(current_block))

    classified_blocks = []

    for i, block in enumerate(blocks, start=1):
        lines = block.splitlines()
        first_line = lines[0].strip()
        nonempty_lines = [ln.strip() for ln in lines if ln.strip()]

        if re.match(r"^\s*#{1,6}\s+", first_line):
            block_type = "header"

        elif all(ln.startswith("- ") for ln in nonempty_lines):
            joined = " ".join(nonempty_lines).lower()

            if re.search(r"\bxp\b|\bcr\d*\b|\(\d+\s*xp", joined):
                block_type = "combat_stats"
            elif re.search(
                r"\bgp\b|\bdagger\b|\bscimitar\b|\bhorses?\b|\bfood\b|\bdrink\b|\bhousehold\b",
                joined
            ):
                block_type = "loot"
            else:
                block_type = "list"

        elif re.search(r"/r\s*\(", block):
            block_type = "xp_accounting"

        else:
            block_type = "narrative"

        if block_type in {"header", "narrative"}:
            classified_blocks.append(
                {
                    "source_id": source_id,
                    "chunk_id": chunk_id,
                    "chunk_start_line": start_line,
                    "chunk_end_line": end_line,
                    "block_num": i,
                    "block_type": block_type,
                    "text": block,
                }
            )

    print("=" * 100)
    print(f"{source_id} | chunk {chunk_id} | lines {start_line}-{end_line}")
    print("=" * 100)

    for b in classified_blocks:
        print(f"\n[b{b['block_num']}] {b['block_type']}")
        print("-" * 80)
        print(b["text"])

    print("\n")

Blackburn-related session_notes chunks: 27

_local/session_notes/IWTC session notes 1-50.md | chunk 169534 | lines 642-649

[b1] header
--------------------------------------------------------------------------------
### Set-up

[b2] narrative
--------------------------------------------------------------------------------
The 4 veterans and scout, David Bowie, return to the Abbey. The landslide is cleared enough. They heard about the orcs and evacuations. The village of Blackburn (on the other side of the landslide) wants guidance. A hunters reports sighting of a troll up in the mountain. Blackburn leaders aren’t sure whether to send fighters to the Abbey to muster for the orcs or to stay and defend against possible trolls. They don’t agree on whether there really is a troll.

[b3] narrative
--------------------------------------------------------------------------------
There are three farming settlements around Blackburn. Blackburn itself mostly has livestock and some infrastructure

## Phase B3: start mapping from the beginning

In [100]:
# -------------------------------------------------------------------
# Phase B3: read chunk by index RANGE (sequential navigation)
# -------------------------------------------------------------------
LAST_PHASE_RUN = "B3"

from itertools import islice

# target source file
sample_source = "_local/pbp_transcripts/PbP10 - The Second Camp.md"

# build ordered chunk list for this file
df_chunks_for_file = (
    DF_CHUNK_TO_ENTITIES
    .loc[DF_CHUNK_TO_ENTITIES["source_id"] == sample_source]
    .sort_values("chunk_start_line")
    .reset_index(drop=True)
)

# ---- choose range of chunks to read ----
start_idx = 0   # inclusive
end_idx = 5     # inclusive

# safety checks
max_idx = len(df_chunks_for_file) - 1
if start_idx < 0 or end_idx > max_idx:
    raise IndexError(f"Range {start_idx}-{end_idx} out of bounds (0-{max_idx})")

if start_idx > end_idx:
    raise ValueError("start_idx must be <= end_idx")

# load file once (important for efficiency)
source_path = WORLD_ROOT / sample_source
with source_path.open("r", encoding="utf-8") as f:
    all_lines = list(f)

# loop through range
for idx in range(start_idx, end_idx + 1):
    row = df_chunks_for_file.iloc[idx]

    chunk_id = int(row["chunk_id"])
    start_line = int(row["chunk_start_line"])
    end_line = int(row["chunk_end_line"])

    lines = all_lines[start_line - 1:end_line]

    print("=" * 80)
    print(f"{sample_source} | chunk {chunk_id} | idx {idx} | lines {start_line}-{end_line}")
    print("-" * 80)
    print("".join(lines))
    print("=" * 80)

_local/pbp_transcripts/PbP10 - The Second Camp.md | chunk 169186 | idx 0 | lines 1-5
--------------------------------------------------------------------------------
* ### **Shworn** **7/3/25, 6:12 PM**

* "although a Spencer is a greater threat i still think we should not let the current threats live," Shworn then glances at both piers and Victor  
* 


_local/pbp_transcripts/PbP10 - The Second Camp.md | chunk 169187 | idx 1 | lines 6-10
--------------------------------------------------------------------------------
* ### **Bysickle** **7/3/25, 6:46 PM**

* *Henry turns to his brother, letting out a sigh as the fatigue from the constant squabbling over Shworn’s proactive attitude when it comes to doling out punishment returns.* “That threat is a source of information, Shworn. Feel free to watch him, I guess, but I’m going to go after the girl.”  
* 


_local/pbp_transcripts/PbP10 - The Second Camp.md | chunk 169188 | idx 2 | lines 11-15
-----------------------------------------------